In [1]:
import pandas as pd
import os
import base64

df = pd.read_csv("../data/raw/full_emoji.csv")

selected_platforms = [
    "Apple",
    "Google",
    "Facebook",
    "Windows",
    "Twitter",
    "JoyPixels",
    "Samsung"
]

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1816 entries, 0 to 1815
Data columns (total 15 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   #          1816 non-null   int64
 1   emoji      1816 non-null   str  
 2   unicode    1816 non-null   str  
 3   name       1816 non-null   str  
 4   Apple      1813 non-null   str  
 5   Google     1816 non-null   str  
 6   Facebook   1727 non-null   str  
 7   Windows    1457 non-null   str  
 8   Twitter    1816 non-null   str  
 9   JoyPixels  1816 non-null   str  
 10  Samsung    1724 non-null   str  
 11  Gmail      720 non-null    str  
 12  SoftBank   476 non-null    str  
 13  DoCoMo     251 non-null    str  
 14  KDDI       637 non-null    str  
dtypes: int64(1), str(14)
memory usage: 212.9 KB


In [3]:
platforms = [
    "Apple",
    "Google",
    "Facebook",
    "Windows",
    "Twitter",
    "JoyPixels",
    "Samsung"
]

for platform in platforms:
    os.makedirs(
        f"../data/processed/{platform}",
        exist_ok=True
    )

In [4]:
import re

def clean_filename(name):
    return re.sub(r'[<>:"/\\|?*]', "_", name)

In [5]:
import os
import base64
import pandas as pd

selected_platforms = [
    "Apple",
    "Google",
    "Facebook",
    "Windows",
    "Twitter",
    "JoyPixels",
    "Samsung"
]

metadata = []

for index, row in df.iterrows():

    # ניקוי שם
    emoji_name = clean_filename(
    row["name"].replace(" ", "_")
)

    for platform in selected_platforms:

        # אם חסר - לדלג
        if pd.isna(row[platform]):
            continue

        try:
            image_data = row[platform]

            # split base64
            _, encoded = image_data.split(",", 1)

            # decode
            image_bytes = base64.b64decode(encoded)

            # שם קובץ ייחודי
            filename = f"{emoji_name}_{index}.png"

            # נתיב שמירה
            save_path = f"../data/processed/{platform}/{filename}"

            # save image
            with open(save_path, "wb") as f:
                f.write(image_bytes)

            # save metadata
            metadata.append({
                "file_name": filename,
                "platform": platform,
                "emoji": row["emoji"],
                "emoji_name": row["name"]
            })

        except Exception as e:
            print(f"Error row {index}, {platform}: {e}")

In [6]:
metadata_df = pd.DataFrame(metadata)

metadata_df.head()

,file_name,platform,emoji,emoji_name
0,grinning_face_0.png,Apple,😀,grinning face
1,grinning_face_0.png,Google,😀,grinning face
2,grinning_face_0.png,Facebook,😀,grinning face
3,grinning_face_0.png,Windows,😀,grinning face
4,grinning_face_0.png,Twitter,😀,grinning face


In [7]:
metadata_df.to_csv(
    "../data/processed/metadata.csv",
    index=False
)

print("Metadata saved")

Metadata saved


In [8]:
print(len(metadata_df))

12169


In [9]:
metadata_df["platform"].value_counts()

platform
Google       1816
Twitter      1816
JoyPixels    1816
Apple        1813
Facebook     1727
Samsung      1724
Windows      1457
Name: count, dtype: int64

In [10]:
print(len(metadata_df))

12169


In [11]:
import os

print(os.listdir("../data/processed/Apple")[:5])

['1st_place_medal_1017.png', '2nd_place_medal_1018.png', '3rd_place_medal_1019.png', 'abacus_1167.png', 'AB_button_(blood_type)_1480.png']


In [12]:
print(len(metadata))

12169


In [13]:
# בודקים אם נשמרו שמות עם :
bad_files = []

for _, row in pd.DataFrame(metadata).iterrows():
    if ":" in row["file_name"]:
        bad_files.append(row["file_name"])

print("Files with colon:", len(bad_files))
print(bad_files[:10])

Files with colon: 0
[]
